# Edisi 002: Sesak mana udara Jakarta: kemarau atau musim hujan?

**Rubrik:** Kota | **Terbit:** Oktober 2026 | **Sumber utama:** Open-Meteo Air Quality (reanalisis CAMS) | **Waktu baca:** ±4 menit

## Pertanyaan

Tiap kemarau, keluhan "sesak napas" dan "udara kotor" membanjiri linimasa. Sebelum percaya,
saya ukur sendiri: 1.510 hari PM2,5 di Jakarta dari Agustus 2022 sampai September 2026, lalu
tiap hari saya tandai masuk kemarau atau musim hujan. Seberapa besar bedanya, dan seberapa
sering kita sebenarnya menghirup udara di atas batas aman?

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import stats
from statsmodels.stats.proportion import proportion_confint, proportions_ztest

from tools import gaya

gaya.terapkan()
df = pd.read_csv("data/udara-harian-jakarta.csv", parse_dates=["tanggal"])
df["tahun"] = df.tanggal.dt.year
df["bulan"] = df.tanggal.dt.month
df["musim"] = np.select(
    [df.bulan.isin([6, 7, 8, 9]), df.bulan.isin([12, 1, 2])],
    ["kemarau", "hujan"], default="pancaroba")
uji = df.dropna(subset=["pm2_5"])
print("hari sah:", len(uji), "| rentang:", uji.tanggal.min().date(), "s.d.", uji.tanggal.max().date())
df.head(3)

hari sah: 1510 | rentang: 2022-08-05 s.d. 2026-09-22


,tanggal,pm2_5,pm2_5_maks,pm10,us_aqi,jam_terisi,hujan_mm,tahun,bulan,musim
0,2022-08-01,NaN,NaN,NaN,NaN,0,0.0,2022,8,kemarau
1,2022-08-02,NaN,NaN,NaN,NaN,0,1.3,2022,8,kemarau
2,2022-08-03,NaN,NaN,NaN,NaN,0,13.6,2022,8,kemarau


## Data & cara ukur

Yang diukur: partikel halus PM2,5 (berukuran 2,5 mikrometer ke bawah) per jam di titik Jakarta
Pusat dari Open-Meteo Air Quality API (reanalisis CAMS), dirangkum jadi rata-rata harian dengan
syarat minimal 18 jam terisi per hari mengikuti kaidah WHO. Hasilnya 1.510 hari sah dari 5
Agustus 2022 sampai 22 September 2026. Musim ditandai klimatologis: kemarau Juni-September,
musim hujan Desember-Februari, sisanya pancaroba yang dikeluarkan dari perbandingan. Sebagai
verifikasi, curah hujan harian ERA5 memang rata-rata 3,3 mm per hari saat kemarau melawan 9,5
mm saat musim hujan.

Uji kelayakan sumber mencatat siapa yang gugur: ISPU BMKG butuh akun (jalur publiknya 404),
OpenAQ butuh API key, NASA POWER tidak punya PM2,5. Yang lulus hanya CAMS, dan batasnya saya
catat sejak awal: CAMS itu model reanalisis, bukan stasiun ISPU. Resolusi gridnya kasar dan
datanya baru empat tahun, cukup untuk membandingkan musim, tidak untuk tren panjang.

In [2]:
ringkas = uji[["pm2_5", "pm10", "pm2_5_maks"]].describe().T[["count", "mean", "std", "min", "max"]].round(1)
print("verifikasi musim: rata-rata hujan harian (mm)")
print(df.groupby("musim").hujan_mm.mean().round(1).to_string())
print("\nrata-rata PM2,5 per bulan kalender:")
print(df.groupby("bulan").pm2_5.mean().round(1).to_string())
rekor = uji.loc[uji.pm2_5.idxmax()]
print(f"\nhari paling sesak: {rekor.pm2_5:.1f} µg/m³ pada {rekor.tanggal.date()}")
ringkas

verifikasi musim: rata-rata hujan harian (mm)
musim
hujan        9.5
kemarau      3.3
pancaroba    6.7

rata-rata PM2,5 per bulan kalender:
bulan
1     31.2
2     35.4
3     39.3
4     53.2
5     64.6
6     60.0
7     57.7
8     53.0
9     52.9
10    55.4
11    51.3
12    42.7

hari paling sesak: 128.5 µg/m³ pada 2023-05-23


,count,mean,std,min,max
pm2_5,1510.0,49.9,19.7,0.6,128.5
pm10,1510.0,63.6,28.3,0.9,185.5
pm2_5_maks,1510.0,83.1,31.7,0.9,186.7


## Pembedahan 1: seberapa jauh jarak dua musimnya?

Ukuran paling sederhana: bandingkan rata-rata PM2,5 harian kemarau (537 hari) dengan musim hujan
(361 hari). Kemarau **55,6 µg/m³**, musim hujan **36,5 µg/m³**. Selisihnya **19,2 µg/m³**
(CI95% 16,9 s.d. 21,5; uji Welch, p < 0,001). Uji Welch dipilih karena dua musim tidak
diwajibkan punya sebaran yang sama besar.

Yang lebih enak dibayangkan adalah Cohen's d: selisih dua rata-rata dalam satuan simpangan
baku. Nilainya **1,13**, artinya rata-rata kemarau berdiri lebih dari satu simpangan baku di
atas rata-rata musim hujan. Dua sebarannya masih tumpang tindih, tapi pusatnya bergeser jelas.
Dua temuan kecil untuk warna: bulan paling kotor ternyata Mei (64,6 µg/m³), awal kemarau,
bukan puncak Agustus; dan hari paling sesak tercatat 23 Mei 2023 (128,5 µg/m³).

In [3]:
k = uji.loc[uji.musim == "kemarau", "pm2_5"]
h = uji.loc[uji.musim == "hujan", "pm2_5"]
selisih = k.mean() - h.mean()
se = np.sqrt(k.var(ddof=1) / len(k) + h.var(ddof=1) / len(h))
dfw = (k.var(ddof=1) / len(k) + h.var(ddof=1) / len(h)) ** 2 / (
    (k.var(ddof=1) / len(k)) ** 2 / (len(k) - 1) + (h.var(ddof=1) / len(h)) ** 2 / (len(h) - 1))
tc = stats.t.ppf(0.975, dfw)
tw = stats.ttest_ind(k, h, equal_var=False)
sp = np.sqrt(((len(k) - 1) * k.var(ddof=1) + (len(h) - 1) * h.var(ddof=1)) / (len(k) + len(h) - 2))

pd.DataFrame({"nilai": [
    f"{h.mean():.1f} (median {h.median():.1f}, std {h.std():.1f}) | n = {len(h)}",
    f"{k.mean():.1f} (median {k.median():.1f}, std {k.std():.1f}) | n = {len(k)}",
    f"{selisih:.1f} µg/m³ (CI95 {selisih - tc * se:.1f} s.d. {selisih + tc * se:.1f})",
    f"t = {tw.statistic:.2f} p = {tw.pvalue:.2e}",
    f"d = {selisih / sp:.2f}",
]}, index=["musim hujan", "kemarau", "selisih (Welch)", "uji-t Welch", "Cohen's d"])

,nilai
musim hujan,"36.5 (median 33.0, std 17.6) | n = 361"
kemarau,"55.6 (median 53.3, std 16.6) | n = 537"
selisih (Welch),19.2 µg/m³ (CI95 16.9 s.d. 21.5)
uji-t Welch,t = 16.35 p = 1.49e-51
Cohen's d,d = 1.13


In [4]:
def grafik_1(mode):
    fig, ax, fs = gaya.dasar(
        mode,
        "Sesak mana udara Jakarta: kemarau atau musim hujan?",
        "Rata-rata 55,6 vs 36,5 µg/m³. Selisih 19,2 (CI95 16,9 s.d. 21,5); Cohen's d 1,13.",
        "Open-Meteo (reanalisis CAMS), PM2,5 harian Jakarta, Agu 2022-Sep 2026", 898,
    )
    kotak = ax.boxplot([h.values, k.values], tick_labels=["Musim hujan\n(Des-Feb)", "Kemarau\n(Jun-Sep)"],
                       patch_artist=True, widths=0.5, showmeans=True,
                       medianprops=dict(color=gaya.INK, lw=1.6),
                       meanprops=dict(marker="D", markersize=6, markerfacecolor=gaya.KERTAS,
                                      markeredgecolor=gaya.INK, markeredgewidth=1.4),
                       whiskerprops=dict(color=gaya.INK2), capprops=dict(color=gaya.INK2),
                       flierprops=dict(marker="o", markersize=3, markerfacecolor=gaya.INK2,
                                       markeredgecolor="none", alpha=0.55))
    for patch, warna in zip(kotak["boxes"], [gaya.INK2, gaya.AKSEN]):
        patch.set_facecolor(warna)
        patch.set_alpha(0.85)
    ax.set_ylabel("PM2,5 harian (µg/m³)", fontsize=9.5 * fs)
    ax.set_xlabel("Musim (berlian = rata-rata)", fontsize=9.5 * fs)
    ax.grid(True, axis="y")
    ax.grid(False, axis="x")
    gaya.simpan(fig, "01-kontras-musim", mode)


for mode in gaya.MODE:
    grafik_1(mode)

## Pembedahan 2: berapa hari di atas batas aman?

Di sini hasilnya bergantung pada siapa yang memegang penggaris. Pedoman WHO 2021 menetapkan
batas aman PM2,5 24 jam pada **15 µg/m³**; baku mutu nasional lewat ISPU (PermenLHK 14/2020)
memakai **55 µg/m³**, hampir empat kali lebih longgar. Saya hitung dua-duanya, lengkap dengan
interval kepercayaan Wilson.

Menurut batas WHO, nyaris tidak ada hari aman: 97% sampai 100% hari tiap tahun melewatinya, di
kedua musim sekaligus (kemarau 100%, hujan 96%). Menurut batas ISPU, angkanya 20% sampai 45%
hari per tahun, dan di sinilah musimnya terasa: hari di atas 55 µg/m³ muncul **3,0 kali lebih
sering** saat kemarau (44% melawan 15%; CI95% 2,3 s.d. 3,8; uji proporsi z = 9,2; p < 0,001).
Menariknya, batas bawah kategori "sedang" ISPU (15,5 µg/m³) nyaris sama dengan batas WHO.
"Buruk atau tidaknya" udara Jakarta jadi pertanyaan tentang siapa yang memegang penggarisnya.

In [5]:
baris = []
for ambang, nama in [(15, "WHO"), (55, "ISPU")]:
    for th, g in uji.groupby("tahun"):
        kk, nn = int((g.pm2_5 > ambang).sum()), len(g)
        lo, hi = proportion_confint(kk, nn, method="wilson")
        baris.append({"ambang": nama, "tahun": f"{th}{'*' if th in (2022, 2026) else ''}",
                      "persen": 100 * kk / nn, "lo": 100 * lo, "hi": 100 * hi})
per_tahun = pd.DataFrame(baris)

hasil_rasio = {}
for ambang, nama in [(15, "WHO"), (55, "ISPU")]:
    kg = uji[uji.musim == "kemarau"]
    hg = uji[uji.musim == "hujan"]
    k1, n1 = int((hg.pm2_5 > ambang).sum()), len(hg)
    k2, n2 = int((kg.pm2_5 > ambang).sum()), len(kg)
    p1, p2 = k1 / n1, k2 / n2
    se_log = np.sqrt((1 - p2) / k2 + (1 - p1) / k1) if k1 and k2 else np.nan
    r = p2 / p1
    z, pz = proportions_ztest([k2, k1], [n2, n1])
    hasil_rasio[nama] = (100 * p2, 100 * p1, r, np.exp(np.log(r) - 1.96 * se_log),
                         np.exp(np.log(r) + 1.96 * se_log), z, pz)
print("kemarau vs hujan (persen hari melewati ambang):")
for nama, (p2, p1, r, lo, hi, z, pz) in hasil_rasio.items():
    print(f"  {nama}: {p2:.0f}% vs {p1:.0f}% | rasio {r:.1f}x (CI95 {lo:.1f}-{hi:.1f}) z={z:.2f}")
per_tahun.round(1)

kemarau vs hujan (persen hari melewati ambang):
  WHO: 100% vs 96% | rasio 1.0x (CI95 1.0-1.1) z=4.76
  ISPU: 44% vs 15% | rasio 3.0x (CI95 2.3-3.8) z=9.16


,ambang,tahun,persen,lo,hi
0,WHO,2022*,96.6,92.4,98.6
1,WHO,2023,97.8,95.7,98.9
2,WHO,2024,98.6,96.8,99.4
3,WHO,2025,98.9,97.2,99.6
4,WHO,2026*,99.6,97.9,99.9
5,ISPU,2022*,34.2,27.1,42.2
6,ISPU,2023,38.4,33.5,43.4
7,ISPU,2024,41.0,36.1,46.1
8,ISPU,2025,44.9,39.9,50.1
9,ISPU,2026*,20.4,16.0,25.6


In [6]:
def grafik_2(mode):
    fig, ax, fs = gaya.dasar(
        mode,
        "Berapa hari setahun kita hirup udara di atas batas aman?",
        "Batas WHO: 97-100% hari per tahun. Batas ISPU: 20-45% hari. Kemarau 3,0x lebih sering.",
        "Open-Meteo (reanalisis CAMS), Jakarta, 2022-2026; WHO 2021 dan ISPU (PermenLHK 14/2020)", 1510,
    )
    who = per_tahun[per_tahun.ambang == "WHO"].reset_index(drop=True)
    ispu = per_tahun[per_tahun.ambang == "ISPU"].reset_index(drop=True)
    x = np.arange(len(who))
    lebar = 0.36
    ax.bar(x - lebar / 2, who.persen, width=lebar, color=gaya.AKSEN,
           yerr=[who.persen - who.lo, who.hi - who.persen], capsize=4,
           error_kw=dict(ecolor=gaya.INK2, lw=1.2), label="WHO: batas 15 µg/m³")
    ax.bar(x + lebar / 2, ispu.persen, width=lebar, color=gaya.INK2,
           yerr=[ispu.persen - ispu.lo, ispu.hi - ispu.persen], capsize=4,
           error_kw=dict(ecolor=gaya.INK2, lw=1.2), label="ISPU: batas 55 µg/m³")
    for i in range(len(who)):
        ax.text(i - lebar / 2, who.persen.iloc[i] + 2.5, f"{who.persen.iloc[i]:.0f}".replace(".", ",") + "%",
                ha="center", fontsize=8.5 * fs, color=gaya.AKSEN, fontweight="bold")
        ax.text(i + lebar / 2, ispu.persen.iloc[i] + 2.5, f"{ispu.persen.iloc[i]:.0f}".replace(".", ",") + "%",
                ha="center", fontsize=8.5 * fs, color=gaya.INK2, fontweight="bold")
    ax.set_xticks(x, who.tahun)
    ax.set_ylabel("Hari melewati ambang (% dari hari)", fontsize=9.5 * fs)
    ax.set_xlabel("* tahun tidak penuh (2022 s.d. Sep; 2026 s.d. Sep)", fontsize=8.5 * fs)
    ax.set_ylim(0, 138)
    leg = ax.legend(loc="upper center", ncol=2, frameon=False, fontsize=9 * fs)
    for t in leg.get_texts():
        t.set_color(gaya.INK)
    ax.grid(True, axis="y")
    ax.grid(False, axis="x")
    gaya.simpan(fig, "02-hari-di-atas-batas", mode)


for mode in gaya.MODE:
    grafik_2(mode)

## Uji kekokohan: bagaimana bila pilihan analisis diubah?

Sumber pembanding kedua untuk PM2,5 tidak ditemukan saat uji kelayakan, jadi yang diuji di
sini bukan silang sumber, melainkan kekokohan terhadap pilihan analisis. Pertama, uji
Mann-Whitney yang tidak berasumsi sebaran normal memberi p < 0,001, sama kuatnya dengan Welch.
Kedua, 2022 dan 2026 yang tidak penuh saya buang; selisihnya justru melebar ke 21,6 µg/m³
(d = 1,22). Kesimpulan "kemarau lebih sesak" tidak bergantung pada dua pilihan itu.

In [7]:
mw = stats.mannwhitneyu(k, h, alternative="two-sided")
kutuh = uji[~uji.tahun.isin([2022, 2026])]
k2 = kutuh.loc[kutuh.musim == "kemarau", "pm2_5"]
h2 = kutuh.loc[kutuh.musim == "hujan", "pm2_5"]
t2 = stats.ttest_ind(k2, h2, equal_var=False)
sp2 = np.sqrt(((len(k2) - 1) * k2.var(ddof=1) + (len(h2) - 1) * h2.var(ddof=1)) / (len(k2) + len(h2) - 2))
print(f"Mann-Whitney (tanpa asumsi normal): U = {mw.statistic:.0f}, p = {mw.pvalue:.2e}")
print(f"Welch tanpa 2022 & 2026: selisih {k2.mean() - h2.mean():.1f} µg/m³, d = {(k2.mean() - h2.mean()) / sp2:.2f}, p = {t2.pvalue:.2e}")

Mann-Whitney (tanpa asumsi normal): U = 154703, p = 6.50e-52
Welch tanpa 2022 & 2026: selisih 21.6 µg/m³, d = 1.22, p = 1.76e-43


In [8]:
def kartu_teks():
    gaya.simpan(gaya.kartu("linkedin", "EDISI 002 · KOTA · OKTOBER 2026",
        "Sesak mana udara Jakarta:\nkemarau atau musim hujan?",
        [(gaya.INK2, "Saya hitung 1.510 hari PM2,5\ndi Jakarta, dari Agustus 2022,\nlalu tiap hari ditandai:\nkemarau atau musim hujan."),
         (gaya.AKSEN, "(jawabannya di dalam)")],
        "Jurnal Eksplorasi Data Keseharian"), "slide-01-pertanyaan", "linkedin", penuh=True)

    gaya.simpan(gaya.kartu("linkedin", "EDISI 002 · KOTA",
        "Datanya dari mana?",
        [(gaya.INK, "Open-Meteo Air Quality (model\nCAMS), PM2,5 per jam titik\nJakarta Pusat, 2022-sekarang."),
         (gaya.INK2, "Model reanalisis, bukan stasiun\nISPU BMKG. Musim diverifikasi\ndengan curah hujan ERA5.\nBMKG, OpenAQ, dan NASA POWER\nditawar lebih dulu: butuh\nkunci.")],
        "Jurnal Eksplorasi Data Keseharian"), "slide-02-data", "linkedin", penuh=True)

    gaya.simpan(gaya.kartu("linkedin", "EDISI 002 · KOTA",
        "Temuan, batas, dan sumber",
        [(gaya.AKSEN, "Benar: kemarau lebih sesak.\nPM2,5 +19,2 µg/m³ dan hari\ndi atas ISPU 3,0x lebih sering."),
         (gaya.INK2, "Batas: model CAMS, bukan stasiun\nISPU; satu titik grid; jendela\n4 tahun, 2026 masih parsial."),
         (gaya.INK2, "Sumber: Open-Meteo (CAMS, ERA5);\nWHO 2021; ISPU PermenLHK 14/2020.")],
        "Jurnal Eksplorasi Data Keseharian"), "slide-05-batas", "linkedin", penuh=True)


kartu_teks()
print("karousel:", sorted(p.name for p in Path("linkedin/gambar").glob("*.png")))

karousel: ['01-kontras-musim.png', '02-hari-di-atas-batas.png', 'slide-01-pertanyaan.png', 'slide-02-data.png', 'slide-05-batas.png']


## Temuan

> **Benar: udara Jakarta lebih sesak saat kemarau, dengan PM2,5 19,2 µg/m³ lebih tinggi dari
> musim hujan dan hari melewati batas ISPU 3,0 kali lebih sering.**

Ukuran efeknya besar (Cohen's d 1,13) dan kokoh di tiga cara hitung berbeda. Tapi temuan yang
paling menohok justru soal garis batas: menurut pedoman WHO, 97% sampai 100% hari di Jakarta,
musim apa pun, termasuk tidak aman. Baku mutu nasional yang hampir empat kali lebih longgar
membuat masalah yang sama terlihat jauh lebih kecil.

## Batas & cara reproduksi

Yang tidak boleh disimpulkan dari edisi ini: penyebab dan sumber polusinya. Asap kendaraan,
industri, pembakaran lahan, dan arah angin semuanya bercampur; menguraikannya butuh data emisi
dan penelitian sendiri. Yang juga harus diingat: CAMS reanalisis bukan stasiun ISPU BMKG,
gridnya kasar, hanya satu titik untuk seluruh Jakarta, dan jendelanya baru empat tahun. Angka
2026 yang lebih rendah sebagian karena datanya berhenti di September, belum melewati
Oktober yang kotor.

```bash
python3 data_scraping.py
python3 -m nbconvert --to notebook --execute --inplace analisis.ipynb
```

## Sumber Data

- Open-Meteo Air Quality API (reanalisis CAMS), titik -6,18 / 106,83, diakses 22 September 2026.
- Open-Meteo Historical Weather API (ERA5) untuk verifikasi curah hujan, diakses 22 September 2026.
- Pedoman kualitas udara WHO 2021 (PM2,5 24 jam = 15 µg/m³); baku mutu ISPU, PermenLHK No. 14 Tahun 2020 (PM2,5 24 jam = 55 µg/m³).
- Data olahan: `data/udara-harian-jakarta.csv`. Kode pengambilan: `data_scraping.py`.
- Data mentah apa adanya: `data/mentah/`.